# EIGSEP Horizons

Aaron Parsons

This notebook uses digital elevation models of Marjum canyon and proposed antenna locations to calculate the distance to surrounding terrain (saved in horizon_models_v000.npz), with NaNs above the horizon. These are used in other notebooks for taking into account horizon effects and for modeling terrain scattering.

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from eigsep_terrain.marjum_dem import MarjumDEM as DEM
import eigsep_terrain as et
import eigsep_terrain.utils as etu
import tqdm
import healpy

%matplotlib widget

In [ ]:
CACHE_FILE = 'marjum_dem.npz'
dem = DEM(cache_file=CACHE_FILE, backend='numba')

In [ ]:
# Known sites
dem['1P'] = np.array([1648, 2024, 1796])  # Site 1, index 0, 114 m off ground
dem['1E'] = dem.interp_alt(1790, 1915, return_vec=True)  # Site 1, East anchor
dem['1W'] = dem.interp_alt(1518, 2124, return_vec=True)  # Site 1, West anchor

#dem['2P'] = np.array([1740, 2264, 1833])  # Site 2, index 2, 124 m off ground
#dem['2E'] = dem.interp_alt(1920, 2240, return_vec=True)  # Site 2, East anchor
#dem['2W'] = dem.interp_alt(1538, 2291, return_vec=True)  # Site 2, West anchor

In [ ]:
e,n,u = dem.latlon_to_enu(39.247907, -113.402715)
u = dem.interp_alt(e, n)
print(e, n, u)
fig, ax = plt.subplots()
et.plot.terrain_plot(dem, ax=ax, decimate=4, vmin=u-200, vmax=u+200)
plt.plot([e], [n], '+', color='k')

In [ ]:
# Calculate the horizon angle of the terrain surrounding the antenna
horizon_angles = {}
horizon_pnts = {}
#for k in ('1', '2'):
for k in ('1'):
    pnt = dem[k + 'P']
    for delta_h in (0,):
        _pnt = pnt + np.array([0, 0, delta_h])
        horizon_angles[k+' '+str(delta_h)], horizon_pnts[k+' '+str(delta_h)] = dem.calc_horizon(*_pnt, n_az=512)

In [ ]:
fig, ax = plt.subplots()
et.plot.terrain_plot(dem, ax=ax, decimate=4, vmin=u-200, vmax=u+200)
for k in horizon_angles.keys():
    color = ax._get_lines.get_next_color()
    plt.plot(horizon_pnts[k][1], horizon_pnts[k][0], '.-', color=color)
    plt.plot([dem[k[0] + 'P'][0]], [dem[k[0] + 'P'][1]], '+', color=color)
_ = plt.title('Horizon Points')

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True, figsize=(6,4))
axes[0].set_title('Horizon Angle from Antenna Site')
for k in horizon_angles.keys():
    if k.startswith('1'):
        ax = axes[0]
    else:
        ax = axes[1]
    ax.fill_between(np.linspace(0, 360, horizon_angles[k].size, endpoint=False), np.rad2deg(horizon_angles[k]), 0, alpha=0.2, label=k)
axes[1].set_xlabel('Azimuthal Angle [deg]')
for ax in axes:
    ax.legend()
    ax.set_ylabel('Horizon Angle [deg]')
    _ = ax.grid()

In [ ]:
NDRAWS = 25
SPATIAL_STD = 0.5  # m
e0, n0, u0 = dem['1P']
u_gnd = dem.interp_alt(e0, n0)
centers = np.empty((NDRAWS + 1, 3), dtype=np.float32)
heights = np.empty((NDRAWS + 1, 3), dtype=np.float32)
centers[0] = (e0, n0, u0)
heights[0] = centers[0, 2] - u_gnd
nside_full = 128
r = np.empty((NDRAWS + 1, healpy.nside2npix(nside_full)), dtype=np.float32)
r[0] = dem.ray_trace(centers[0], nside=nside_full)
for i in range(NDRAWS):
    centers[i + 1] = centers[0] + np.random.normal(scale=SPATIAL_STD, size=3)
    heights[i + 1] = centers[i + 1, 2] - u_gnd
    r[i + 1] = dem.ray_trace(centers[i + 1], nside=nside_full)

In [ ]:
# 1.5 m ->  2.91 K (Tsky=2000, Tgnd=300)
# 0.5 m ->  0.78 K

In [ ]:
T_iso = np.mean(np.where(np.isnan(r), 2e3, 300), axis=-1)
print(np.std(T_iso))
plt.figure()
plt.plot(T_iso)

In [ ]:
#healpy.mollview(h.map)
healpy.mollview(np.log10(r[0]), cmap='plasma')
#healpy.mollview(T[-1], cmap='plasma')
#healpy.mollview((T_sky_full[:, 180]), cmap='plasma')

In [ ]:
#np.savez('horizon_models_v000.npz', r=r, heights=heights, centers=centers, nside=nside_full)

In [ ]:
fig, axes = plt.subplots()
for i, dh in enumerate(heights):
    h, bins = np.histogram(r[i], bins=np.logspace(-1, 3.5, 200))
    axes.semilogy(0.5 * (bins[1:] + bins[:-1]), h, label=dh)
axes.set_title('Angular Area by Radius')
axes.set_xlim(0, 1600)
axes.set_ylabel('Pixel Count')
_ = axes.set_xlabel('Radius [m]')